# nb

> Learn in a notebook (leela or Jupyter): question cards from the tutor.

This module brings the tutor into a notebook. `learn` shows a question card in the output of a cell.
It uses the same `Tutor` and the same `learning` shelf as the web app. Code that you write runs in Leela: a walk step with a notebook check opens there.

In [ ]:
#| default_exp nb

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # and the settings, with the saved prompts, go to a temporary folder

These are the imports of the module.

In [ ]:
#| export
import html
from fastcore.utils import *
from abhyasa.model import DEPTHS
from abhyasa.tutor import Tutor, Session
from abhyasa.llm import route_ask
from abhyasa import demo as _demo   # registers the demo hook of the tutor

## Use

Leela shows ipywidgets, so the card is a widget in the output of the cell.

```python
from abhyasa.nb import learn
learn('reciprocal-rank-fusion')      # a question card
```

## The tutor

`tutor` makes the tutor of the notebook one time. A call with a store or a model makes a new one.

In [ ]:
#| export
_st = AttrDict(tutor=None)

def tutor(learning=None, ask=None) -> Tutor:
    "The tutor of the notebook, made one time: the default vault and `route_ask`, unless you give others."
    if _st.tutor is None or learning is not None or ask is not None:
        _st.tutor = Tutor(learning or Learning(os.getenv('ABHYASA_VAULT') or os.getenv('VISHALAKSHI_VAULT')), ask or route_ask())
    return _st.tutor

def _esc(s): return html.escape(s or '').replace('\n', '<br>')

## The question card

`Card` has a box for the answer, a box for the think-aloud, a confidence slider and four buttons.

In [ ]:
#| export
class Card:
    "A question card widget over a `Session`: answer, think aloud, confidence, hint, and next."
    def __init__(self, sess:Session):
        import ipywidgets as w
        self.s, self.w = sess, w
        self.q = w.HTMLMath(); self.fb = w.HTMLMath()
        self.ans = w.Textarea(placeholder='Your answer', layout=w.Layout(width='100%', height='70px'))
        self.think = w.Textarea(placeholder='Think aloud: how you got there, what you are unsure of', layout=w.Layout(width='100%', height='90px'))
        self.conf = w.IntSlider(50, 0, 100, description='Confidence')
        b = lambda t, f, st='': (x := w.Button(description=t, button_style=st), x.on_click(lambda _: self._do(f)), x)[-1]
        self.buttons = w.HBox([b('Send', self._send, 'primary'), b('Hint', self._hint), b("I don't know", self._idk), b('Next', self._next)])
        self.demo = w.Output()
        self.box = w.VBox([self.q, self.ans, self.think, self.conf, self.buttons, self.fb, self.demo])
        self._next()

    def _do(self, f):
        try: f()
        except Exception as e: self.fb.value = f'<b>Error</b>: {_esc(str(e))}'

    def _show(self, q):
        if q is None: self.q.value, self.fb.value = '<b>Nothing to learn or review right now.</b>', ''; return
        opts = ''.join(f'<li>{i}. {_esc(o)}</li>' for i, o in enumerate(q.options)) if q.get('options') else ''
        self.q.value = (f"<small>{_esc(q.name)} · {DEPTHS[q.depth]} · {_esc(q.why)}</small><p style='font-size:1.2em'>{_esc(q.question)}</p>"
                        + (f'<ul style="list-style:none">{opts}</ul><small>Answer with the number.</small>' if opts else ''))
        self.ans.value = self.think.value = self.fb.value = ''

    def _send(self):
        r = self.s.answer(self.ans.value, self.think.value, self.conf.value)
        self.fb.value = fb_html(r); self._demo(r)
    def _demo(self, r):
        from IPython.display import display
        self.demo.clear_output()
        if (v := demo_view(r, self.s.tutor.learning)):
            with self.demo: display(v)
    def _hint(self): self.fb.value = f'<i>Hint:</i> {_esc(self.s.hint())}'
    def _idk(self): r = self.s.answer(dont_know=True); self.fb.value = fb_html(r); self._demo(r)
    def _next(self): self._show(self.s.next())
    def _ipython_display_(self):
        from IPython.display import display
        display(self.box)

`fb_html` shows a result in the card. `learn` starts a session in a card.

In [ ]:
#| export
def fb_html(r) -> str:
    "A result as HTML."
    if not r.done: return f"<b>Not yet.</b> {_esc(r.feedback)}<br><small>{r.tries_left} more tries.</small>"
    parts = [f"<b>{r.outcome}</b>", _esc(r.feedback), _esc(r.lesson), f"<small>Floor now {DEPTHS[r.floor]}. Press Next.</small>"]
    return '<br><br>'.join(p for p in parts if p)

def learn(focus:str=None, subject:str=None, limit:int=12) -> Card:
    "A question card for a session, on one concept when you name it."
    return Card(Session(tutor(), focus=focus, subject=subject, limit=limit))

After a miss on a concept that has a demo, the card shows the demo under the feedback. `demo_view` gives the demo as a notebook output. It uses `show` from rupaka, and leela draws the output in a sandboxed frame.

In [ ]:
#| export
def demo_view(r, learning):
    "The demo of result `r` as a notebook output, or None."
    from rupaka.render import show
    return show(d.spec) if r.get('demo') and (d := learning.demo(r.demo)) else None

A result with a demo gives an output with the rupaka mime and a full HTML page. A result without one gives nothing.

In [ ]:
from abhyasa.store import temp_learning
from rupaka.spec import gallery
from rupaka.render import MIME

In [ ]:
dl = temp_learning(); dl.put('Vowel sandhi', 'Vowels join by rule.', 'sanskrit')
dl.put_demo(gallery()['vowel-sandhi'], 'vowel-sandhi')
mb = demo_view(AttrDict(demo='vowel-sandhi'), dl)._repr_mimebundle_()
assert MIME in mb and '<script' in mb['text/html']
test_eq(demo_view(AttrDict(correct=True), dl), None)

A fake model writes the questions. The tutor of the notebook uses a temporary shelf.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake

In [ ]:
lr = temp_learning()
lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml')
fake = Fake({'Teach the smallest step': 'Sum 1/(k+r). Check: what if k is large?'}, dflt=dict(question='Why ranks?', answer='Comparable.', rubric=['comparable']))
t = tutor(lr, fake)

`learn` shows a card with the first question.

In [ ]:
c = learn('reciprocal-rank-fusion')
assert 'Why ranks?' in c.q.value

The default model of the notebook tutor is the routed one. This test swaps `route_ask` for a marker, so it makes no model call.

In [ ]:
real, mark = route_ask, lambda p, sp='': 'ok'
route_ask = lambda: mark
try:
    _st.tutor = None
    assert tutor(lr).ask is mark
finally: route_ask = real
t = tutor(lr, fake)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()